# Embedding-Based Job Search - Exercise

**Business problem:** rank advertisements that match a user's query.

A model supports a decision; it does not make the business decision for us.

## Learning objectives

- Compare BM25, retrieval-trained embeddings and rank fusion.
- Compute relevance metrics with explicit draft judgments.
- Connect retrieval quality to requirements and user usefulness.

## Dataset and practical scope

24 approved authentic German excerpts. Titles are anonymized and are not indexed. Four development and eight test queries have assistant-drafted judgments needing review. T03 was revised after inspecting earlier results: this is a revised teaching comparison, not untouched holdout evidence or recall over all 5,011 full ads.

Complete the `None` values marked `# TODO` before running the affected cells. Replace the answer prompts with one or two sentences.

Section 7 adds interactive search over the **5,011 real ads** used in Lecture 2. This larger collection has no full-ad relevance judgments; it is separate from the 24-excerpt evaluation above. It is our selected cohort, not the entire source collection.

The larger search section is supplied in full. The exercise keeps its two earlier coding gaps.

## Libraries and settings

Use the prepared Python 3.11 Codespace kernel. No GPU or API key is needed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Minimal data inspection

Only the quoted fragments are searched. The longer criterion describes our draft intent, not additional model input.

In [ ]:
import json
import re
from collections import Counter

panel = json.loads((DATA_ROOT / 'nlp_search_benchmark.json').read_text(encoding='utf-8'))
documents = pd.DataFrame([{'id': item['id'], 'title': item['title'],
                           'text': ' [...] '.join(item['fragments'])}
                          for item in panel['documents']])
queries = panel['queries']
display(documents.head(3))
print('Ads:', len(documents), '| Queries:', len(queries), '| Draft judgments:', panel['judgment_status'])

## 2. Lexical baseline: BM25

The supplied implementation reuses the lecture's term-count, rarity and length formula. It preserves identifiers such as C# and .NET. Zero-score ads are not used to fill an empty list; no exact matches is a valid result.

In [ ]:
def tokens(text):
    return re.findall(r"c\+\+|c#|\.net|[^\W_]+(?:[-'][^\W_]+)*", text.casefold())

def build_bm25(texts):
    counts = [Counter(tokens(text)) for text in texts]
    lengths = np.array([sum(row.values()) for row in counts], dtype=float)
    if not counts or lengths.mean() == 0:
        raise ValueError('BM25 needs a nonempty collection containing searchable words.')
    document_frequency = Counter(term for row in counts for term in row)
    return counts, lengths, document_frequency

def bm25_scores(query, index):
    counts, lengths, document_frequency = index
    scores = np.zeros(len(counts))
    for term in set(tokens(query)):
        frequency = np.array([row[term] for row in counts], dtype=float)
        idf = np.log1p((len(counts) - document_frequency[term] + 0.5)
                      / (document_frequency[term] + 0.5))
        scores += idf * frequency * 2.2 / (
            frequency + 1.2 * (0.25 + 0.75 * lengths / lengths.mean()))
    return scores

def rank_ids(scores, ids, k=3, positive_only=False):
    if len(scores) != len(ids) or not np.isfinite(scores).all():
        raise ValueError('Ranking needs one finite score per document ID.')
    order = sorted(range(len(ids)), key=lambda i: (-scores[i], ids[i]))
    return [ids[i] for i in order if not positive_only or scores[i] > 0][:k]

document_ids = documents['id'].tolist()
small_bm25 = build_bm25(documents['text'])
K, DEPTH = 3, 10
keyword_lists = {query['id']: rank_ids(
    bm25_scores(query['text'], small_bm25), document_ids, DEPTH, True)
    for query in queries}
print('Exact keyword matches for data engineer:', keyword_lists['T03'])

## 3. Encode passages and queries

First use downloads public E5 files; text stays local. Prefixes are model-specific. Normalized vectors make dot product equal cosine. **Coding task:** compute all query-ad scores with `query_vectors @ ad_vectors.T`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(2)
MODEL = 'intfloat/multilingual-e5-small'
REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
search_encoder = SentenceTransformer(MODEL, revision=REVISION, device='cpu')
ad_vectors = search_encoder.encode(['passage: ' + text for text in documents['text']],
                                   normalize_embeddings=True, show_progress_bar=False)
query_vectors = search_encoder.encode(['query: ' + query['text'] for query in queries],
                                      normalize_embeddings=True, show_progress_bar=False)
print('Ad-vector shape:', ad_vectors.shape)
scores = None  # TODO: query-by-ad cosine score matrix
dense_lists = {query['id']: rank_ids(scores[i], document_ids, DEPTH) for i, query in enumerate(queries)}

## 4. Combine candidate lists

**Coding task:** use `1 / (60 + position)` for each list's contribution. Missing results contribute nothing; a result missing from both lists cannot be recovered by fusion.

In [ ]:
def fuse(rankings, k=3):
    votes = {}
    for ranking in rankings:
        for position, ad in enumerate(ranking, 1):
            contribution = None  # TODO: rank contribution
            votes[ad] = votes.get(ad, 0) + contribution
    return sorted(votes, key=lambda ad: (-votes[ad], ad))[:k]

hybrid_lists = {query['id']: fuse([keyword_lists[query['id']], dense_lists[query['id']]])
                for query in queries}

## 5. Development evaluation

Precision divides by K even if a list has fewer hits. Recall divides by the number of draft-relevant excerpts. Reciprocal rank uses the first relevant result; averaging it gives MRR. Review the supplied code and judgments.

In [ ]:
def metrics(ranking, relevant, k=3):
    positions = [i for i, ad in enumerate(ranking[:k], 1) if ad in relevant]
    return {'P@3': len(positions) / k,
            'R@3': len(positions) / len(relevant),
            'RR@3': 1 / positions[0] if positions else 0.}

rows = []
recipes = {'BM25': keyword_lists, 'E5': dense_lists, 'RRF': hybrid_lists}
for query in queries:
    for name, lists in recipes.items():
        rows.append({'query': query['id'], 'split': query['split'], 'recipe': name,
                     **metrics(lists[query['id']], set(query['relevant']))})
measurements = pd.DataFrame(rows)
display(measurements[measurements['split'] == 'development']
        .groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean().rename(columns={'RR@3': 'MRR@3'}).round(3))

### Interpret the result

Why can a high score or agreement between methods still return a poor match?

**Your interpretation:** Replace this sentence with your answer.

## 6. Fixed-recipe test comparison

Keep K=3, candidate depth 10 and RRF constant 60 fixed. Do not retune after this table. These eight queries are not a pristine holdout.

In [ ]:
test = measurements[measurements['split'] == 'test']
display(test.groupby('recipe')[['P@3', 'R@3', 'RR@3']].mean()
        .rename(columns={'RR@3': 'MRR@3'}).round(3))

## Interpretation: read evidence, not just the mean

Inspect the same query under every recipe. MATCH means our draft criterion is supported in the excerpt, not verified suitability.

In [ ]:
selected = next(query for query in queries if query['id'] == 'T03')
print('Query:', selected['text'])
print('Draft criterion:', selected['criterion'])
for name, rankings in recipes.items():
    ids = rankings[selected['id']][:K]
    hits = documents.set_index('id').loc[ids].copy()
    hits['draft match'] = [ad in selected['relevant'] for ad in ids]
    print(name)
    display(hits[['title', 'text', 'draft match']])

### Interpret the result

Which recipe deserves further testing? What would a deployment evaluation add?

**Your interpretation:** Replace this sentence with your answer.

## 7. Search the larger real-ad collection

Now apply the same three methods to **all 5,011 real ads from Lecture 2**. The authorised compressed file is already supplied; there is no new data download. Unlike the 24 short excerpts, these are full cleaned descriptions.

We combine Lecture 2's date partitions into one searchable snapshot. We are not training a new category model: E5 stays fixed, and category labels are only displayed beside results. This does not evaluate how search would perform on future ads.

The earlier evaluation keeps its own documents, query lists and judgments. Its IDs and labels must **not** be attached to these larger-collection results. Here we inspect results rather than calculate precision or recall.

In [ ]:
LARGE_DATA_PATH = DATA_ROOT / 'nlp_real_job_categories.csv.gz'
large_documents = pd.read_csv(LARGE_DATA_PATH)
required = ['teaching_id', 'text', 'category']
if not set(required).issubset(large_documents.columns):
    raise ValueError('The real-ad file needs teaching_id, text and category columns.')
if large_documents[required].isna().any().any():
    raise ValueError('The real-ad collection contains missing IDs, text or categories.')
if (not large_documents['teaching_id'].is_unique
        or not large_documents['text'].map(lambda text: isinstance(text, str) and bool(text.strip())).all()):
    raise ValueError('The real-ad collection needs unique IDs and nonempty descriptions.')
large_documents = large_documents.rename(columns={'teaching_id': 'id'})
large_ids = large_documents['id'].tolist()
large_bm25 = build_bm25(large_documents['text'])
print(f'Indexed {len(large_documents):,} real ads for keyword search.')
print('Category labels are displayed for orientation, not used to rank ads.')

### Create the ad vectors once

The slow step is reading every description with E5. We process small batches and show a progress bar. Then we save the vectors in a local **cache**: a file that lets us reuse the work next time.

The code checks that the saved vectors match the data, model and settings before reusing them. Changed settings trigger a clearly reported rebuild. A damaged cache raises an error; delete `00_Data/nlp-search-cache/real-job-e5-vectors.npz` and rerun this cell if necessary. The cache is ignored by Git; do not publish it as anonymous data.

**Important input difference:** BM25 searches the complete cleaned description. E5 reads only the beginning, up to its token limit. The code reports how many ads exceed that limit. A requirement near the end can affect BM25 but be invisible to E5, so this is not a controlled comparison of model architecture alone. Splitting long ads into passages is an optional later extension.

No vectors are trained on the ads. The existing E5 model turns them into numbers; the cache stores those numbers, not new model weights. Prepare this step before the lecture demonstration.

In [ ]:
import hashlib
import time
from importlib.metadata import version

CACHE_PATH = DATA_ROOT / 'nlp-search-cache' / 'real-job-e5-vectors.npz'
LARGE_BATCH_SIZE = 16
passages = ['passage: ' + text for text in large_documents['text']]
input_lengths = np.array([
    len(search_encoder.tokenizer.encode(text, truncation=False, verbose=False))
    for text in passages
])
input_limit = int(search_encoder.max_seq_length)
print(f'E5 input limit: {input_limit} tokens, including the passage prefix and special markers.')
print(f'Ads exceeding that limit: {int((input_lengths > input_limit).sum()):,}'
      f' / {len(passages):,}. Their endings are not encoded.')

cache_settings = {
    'format': 1,
    'data_sha256': hashlib.sha256(LARGE_DATA_PATH.read_bytes()).hexdigest(),
    'model': MODEL, 'revision': REVISION,
    'input_limit': input_limit, 'passage_prefix': 'passage: ',
    'normalized': True, 'batch_size': LARGE_BATCH_SIZE,
    'dimensions': int(ad_vectors.shape[1]),
    'versions': {package: version(package) for package in
                 ['sentence-transformers', 'transformers', 'torch', 'numpy']},
}

def validate_cached_vectors(vectors, row_count, dimensions):
    if vectors.shape != (row_count, dimensions) or vectors.dtype != np.float32:
        raise ValueError('Unexpected cached vector shape or type. Delete the cache file and rerun.')
    if not np.isfinite(vectors).all() or not np.allclose(
            np.linalg.norm(vectors, axis=1), 1, atol=1e-4):
        raise ValueError('Invalid cached vectors. Delete the cache file and rerun.')

large_vectors = None
if CACHE_PATH.exists():
    with np.load(CACHE_PATH, allow_pickle=False) as saved:
        stored_settings = json.loads(saved['settings'].item())
        if stored_settings == cache_settings:
            large_vectors = saved['vectors']
    if large_vectors is None:
        print('The data, model or settings changed; rebuilding the cached vectors.')
    else:
        validate_cached_vectors(large_vectors, len(large_documents), cache_settings['dimensions'])
        print('Loaded matching cached E5 vectors; no ad encoding needed.')

if large_vectors is None:
    print('Creating E5 vectors. The progress bar shows the batches completed.', flush=True)
    started = time.perf_counter()
    large_vectors = search_encoder.encode(
        passages, batch_size=LARGE_BATCH_SIZE, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True).astype(np.float32, copy=False)
    validate_cached_vectors(large_vectors, len(large_documents), cache_settings['dimensions'])
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = CACHE_PATH.with_suffix('.tmp')
    with temporary_path.open('wb') as temporary:
        np.savez_compressed(temporary, vectors=large_vectors,
                            settings=json.dumps(cache_settings, sort_keys=True))
    temporary_path.replace(CACHE_PATH)
    print(f'Encoding finished in {time.perf_counter() - started:.1f} seconds; vectors saved locally.')

print('Vector matrix:', large_vectors.shape,
      f'| Memory: {large_vectors.nbytes / 1024**2:.2f} MiB')

### Try your own query

Edit `user_query` in the code cell below, then rerun **only that code cell** to search again. There is no need to re-encode the ads.

Each method shows up to five suggestions. Hybrid search combines the first 20 candidates from each method using the same reciprocal-rank rule as before. BM25 can return fewer results or none; E5 still finds the closest vectors even when none is a good match.

Read the descriptions, not just the ranks. The displayed preview is only the beginning of each ad and may omit the evidence. To inspect one full description locally, use `print(large_lookup.loc['JOB-00001', 'text'])`, replacing the ID with a returned teaching ID. Do not copy additional descriptions into publicly shared answers.

**There are no automatic quality scores here.** The 24-excerpt judgments do not cover this collection. Unjudged ads are not automatically irrelevant; categories are not search-relevance labels. Precision@K would require judging the returned ads. Recall@K would require knowing which other ads in the collection are relevant.

In [ ]:
user_query = 'Data Engineer mit Python und SQL'
LARGE_K, LARGE_DEPTH = 5, 20

if not isinstance(user_query, str) or not user_query.strip():
    raise ValueError('Enter a nonempty search query.')
if not 1 <= LARGE_K <= LARGE_DEPTH <= len(large_documents):
    raise ValueError('Use 1 <= LARGE_K <= LARGE_DEPTH <= the number of indexed ads.')
large_keyword_scores = bm25_scores(user_query, large_bm25)
query_input = 'query: ' + user_query.strip()
query_length = len(search_encoder.tokenizer.encode(query_input, truncation=False, verbose=False))
if query_length > search_encoder.max_seq_length:
    raise ValueError('This query exceeds the E5 input limit; shorten it before comparing results.')
large_query_vector = search_encoder.encode(
    [query_input], normalize_embeddings=True, show_progress_bar=False)
large_dense_scores = (large_query_vector @ large_vectors.T)[0]
large_keyword_ids = rank_ids(large_keyword_scores, large_ids, LARGE_DEPTH, True)
large_dense_ids = rank_ids(large_dense_scores, large_ids, LARGE_DEPTH)
large_rankings = {
    'BM25': large_keyword_ids[:LARGE_K],
    'E5': large_dense_ids[:LARGE_K],
    'Hybrid (RRF)': fuse([large_keyword_ids, large_dense_ids], k=LARGE_K),
}
print('Query:', user_query)
print('No relevance metrics: these full-ad results have not been judged.')
large_lookup = large_documents.set_index('id')
for name, ranking in large_rankings.items():
    print(f'\n{name}: {len(ranking)} suggestions')
    if not ranking:
        print('No keyword matches. The list is deliberately left empty.')
        continue
    hits = large_lookup.loc[ranking, ['category', 'text']].copy()
    hits.insert(0, 'rank', np.arange(1, len(hits) + 1))
    hits['description preview'] = hits.pop('text').str.slice(0, 260)
    display(hits)

### Interpret the larger search results

1. For one query, explain one difference between the BM25, E5 and hybrid lists using evidence in the descriptions.
2. Which required condition would you verify before showing a result to a job seeker?
3. Why can we not reuse the earlier recall scores for this collection?

**Your interpretation:** Replace this sentence with your answer.

## Summary and business implications

Offline relevance is not online user benefit. Missing excerpt evidence is not proof that the whole ad is irrelevant. Do not silently revise labels to make a model win.

## Limitations

This is a small teaching example, not evidence of deployment performance. Keep evaluation separate from fitting and model selection. Check errors and the user's decision, not only a headline metric.

The larger search uses 5,011 real ads but has no reviewed full-ad relevance judgments. BM25 sees full cleaned descriptions; E5 truncates long input. Do not interpret the larger candidate pool or a high similarity score as proof of better search quality. Redaction and embeddings do not guarantee anonymity.

## Further exercises

- Inspect C# .NET (T07) and the shift query (T08).
- Can recall change while the first relevant rank stays fixed?
- Optional: compare plain pooled BERT, chunking or reranking on fresh reviewed data.
- Choose a fresh query for the larger collection, judge the pooled top results from all three methods, and report unjudged results explicitly. Do not claim full-collection recall from this limited judging exercise.

## References

**Required reading:** https://www.sbert.net/examples/applications/semantic-search/README.html
and https://huggingface.co/intfloat/multilingual-e5-small
**Instructor references:** Reimers & Gurevych (2019), *Sentence-BERT*, https://aclanthology.org/D19-1410/;
Wang et al. (2024), *Multilingual E5 Text Embeddings*, https://arxiv.org/abs/2402.05672;
Cormack et al. (2009), *Reciprocal rank fusion*, https://doi.org/10.1145/1571941.1572114.
**Official BM25 documentation:** https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.